# Day 7 · Transformer assignment

**Core work: 30–40 minutes, after setup.** Work alone or in pairs. This notebook runs independently of the demonstration; it does not rely on variables from another kernel. Use the [guided lab](../Transformers_what_can_they_do_.ipynb) as a reference.

**Submit:** this notebook with your completed code, your own outputs and short written answers. Required evidence is a task/model/architecture table, token-count comparison, sentiment stress test, two bounded sampled continuations, a greedy comparison and one limitation. Exact generated text need not match another learner's.

Cells marked `TODO` are deliberately incomplete. Running the starter notebook prints reminders rather than pretending the exercises are solved. Short hints follow each task. Instructor solutions are in a separate file and are not needed to begin.

## Setup - do this before the session

Use a **fresh Python 3.12 environment** with **PyTorch 2.6.0 CPU** and **Transformers 4.57.1**. These are the course pins, not a claim that they are the newest releases. Do not upgrade to v5 during the session: some older pipeline tasks, including summarization and translation, were removed from that API. A v5 migration needs different task code and a new smoke test. [Versioned pipeline documentation](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines) · [Current translation model instructions](https://huggingface.co/Helsinki-NLP/opus-mt-fr-en)

The first model load downloads public weights and tokenizer files. The three core weight files total about **0.89 GB**, excluding packages and cache overhead. Allow **at least 3 GB of free disk space** for the core environment/cache. A recent laptop with **4 CPU cores and 8 GB RAM** is the planning assumption. Budget **5–20 minutes for first downloads and setup**, and roughly **1–5 minutes of computation after caching**; these are planning estimates, not benchmarks or guarantees. Network speed, package installation and hardware can change them substantially.

Examples use invented, non-sensitive text. The code runs model inference locally; downloading weights requires a Hub connection. Do not paste client material into a public Hub widget. No Hugging Face login or paid inference API is required for these public checkpoints.

In [1]:
# Optional one-time installation in a fresh notebook kernel, then restart it:
# %pip install "torch==2.6.0" --index-url https://download.pytorch.org/whl/cpu
# %pip install "transformers==4.57.1"
# On macOS, use this torch install line instead of the CPU-index line:
# %pip install "torch==2.6.0"

By the way, I only have Python 3.14 on my laptop and torch 2.6.0 doesn't instal on it. So I used torch 2.15 (CPU) and transformers 4.57.6, which is still v4. I changed the version check in the next cell to print a note instead of stopping.

In [2]:
import os
import platform
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version

OFFLINE = False  # Change before imports; restart the kernel after changing it.
if OFFLINE:
    os.environ['HF_HUB_OFFLINE'] = '1'
os.environ.setdefault('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
LOCAL_FILES_ONLY = OFFLINE or os.environ.get('HF_HUB_OFFLINE') == '1'

def require_versions(required):
    for package, expected in required.items():
        try:
            actual = version(package)
        except PackageNotFoundError as exc:
            raise RuntimeError(f'Install {package}=={expected} in this kernel, then restart it.') from exc
        print(f'{package}: {actual}')
        if actual.split('+')[0] != expected:
            # no torch 2.6.0 for python 3.14, so just print it instead of stopping
            print(f'  note: course pin is {package}=={expected}, running {actual}')

require_versions({'torch': '2.6.0', 'transformers': '4.57.1'})
import torch
from transformers import AutoTokenizer, pipeline, set_seed

torch.set_num_threads(min(4, os.cpu_count() or 1))
SEED = 42
set_seed(SEED)
print('Python:', platform.python_version())
print('Device: CPU; threads:', torch.get_num_threads())
print('HF_HOME:', os.environ['HF_HOME'])
print('Local cache only:', LOCAL_FILES_ONLY)

torch: 2.15.0+cpu
  note: course pin is torch==2.6.0, running 2.15.0+cpu
transformers: 4.57.6
  note: course pin is transformers==4.57.1, running 4.57.6


C:\Users\artem.lubkivskyi\Documents\GitHub\Accenture\Day 7\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.14.7
Device: CPU; threads: 4
HF_HOME: C:\Users\artem.lubkivskyi\.cache\huggingface
Local cache only: False


In [3]:
MODEL_SPECS = {'sentiment': {'task': 'sentiment-analysis',
               'model': 'distilbert/distilbert-base-uncased-finetuned-sst-2-english',
               'revision': '714eb0fa89d2f80546fda750413ed43d93601a13'},
 'mask': {'task': 'fill-mask',
          'model': 'distilbert/distilbert-base-uncased',
          'revision': '12040accade4e8a0f71eabdb258fecc2e7e948be'},
 'generation': {'task': 'text-generation',
                'model': 'distilbert/distilgpt2',
                'revision': '2290a62682d06624634c1f46a6ad5be0f47f38aa'}}

def load_pipeline(name, **task_options):
    spec = MODEL_SPECS[name]
    tokenizer = AutoTokenizer.from_pretrained(
        spec['model'], revision=spec['revision'],
        local_files_only=LOCAL_FILES_ONLY, trust_remote_code=False,
    )
    result = pipeline(
        task=spec['task'], model=spec['model'], tokenizer=tokenizer,
        revision=spec['revision'], framework='pt', device=-1,
        trust_remote_code=False,
        model_kwargs={'use_safetensors': True, 'local_files_only': LOCAL_FILES_ONLY},
        **task_options,
    )
    print(f"{name}: {spec['model']} @ {spec['revision'][:7]}")
    print(type(result.model).__name__, '/', type(result.tokenizer).__name__)
    return result

## Task 1 · Match the parts (5 minutes)

Complete the table. Use the three checkpoints in `MODEL_SPECS`. In the last column, describe the result's meaning rather than just its Python type.

| Task | Checkpoint | Architecture family | What the output means |
|---|---|---|---|
| Sentiment | distilbert-base-uncased-finetuned-sst-2-english | Encoder (DistilBERT) with a classification head | POSITIVE or NEGATIVE for the whole text, and how sure the model is about it |
| Fill-mask | distilbert-base-uncased | Encoder (DistilBERT) with a masked word head | The most likely words for the [MASK] spot, each with a probability |
| Text generation | distilgpt2 | Decoder (GPT-2) | The prompt continued token by token, it sounds ok but nothing is fact checked |

**Explain in 2–3 sentences:** why does selecting a base BERT-like encoder not automatically give us a trained sentiment classifier? What must match between a tokenizer and a checkpoint?

**Success:** all three mappings are complete; your explanation distinguishes architecture, learned weights, output head and tokenizer. **Hint:** read the model card's task and training description; `pipeline` is a wrapper, not an architecture.

**Your answer:** The base BERT was only trained to guess masked words, so it doesn't have a sentiment head. A new head starts random and only works after fine-tuning on labelled sentences, which is what the SST-2 version did. The tokenizer has to be the one the model was trained with (same vocabulary, lowercasing and special tokens), otherwise the token IDs point to the wrong embeddings.

## Task 2 · Make tokens visible (7 minutes)

Load the mask pipeline in the next cell. For `token_text`, compute a whitespace word count, token IDs **without special tokens**, and a token count. Print the token strings too. Then construct one sentence using `unmasker.tokenizer.mask_token` and request its top three completions.

Change one context word and run the mask prediction again. Record the candidate token and score that changed most noticeably. Do not guess a prediction before running it.

**Success:** you show both counts and the actual IDs/token strings; two inputs each contain exactly one valid mask; each result has three candidates. Explain why top-three scores need not sum to one.

**Hint:** `tokenizer(text, add_special_tokens=False)` returns a dictionary containing `input_ids`; `convert_ids_to_tokens` makes those IDs readable. The tokenizer exposes `mask_token`.

In [4]:
unmasker = load_pipeline('mask')
token_text = "Re-tokenization isn't always one word per token."
word_count = len(token_text.split())
token_ids = unmasker.tokenizer(token_text, add_special_tokens=False)['input_ids']
token_count = len(token_ids)
# a finance sentence, curious if it gets "rates"
masked_text = f'The central bank decided to raise interest {unmasker.tokenizer.mask_token} again.'
mask_results = unmasker(masked_text, top_k=3)

if any(value is None for value in [word_count, token_ids, token_count, masked_text, mask_results]):
    print('Complete Task 2 TODOs, then rerun this cell.')
else:
    print('Words:', word_count, 'Tokens:', token_count)
    print('Token IDs:', token_ids)
    print('Token strings:', unmasker.tokenizer.convert_ids_to_tokens(token_ids))
    print(mask_results)
    assert masked_text.count(unmasker.tokenizer.mask_token) == 1
    assert len(mask_results) == 3
    assert token_count == len(token_ids)

Device set to use cpu


mask: distilbert/distilbert-base-uncased @ 12040ac
DistilBertForMaskedLM / DistilBertTokenizerFast


Words: 7 Tokens: 13
Token IDs: [2128, 1011, 19204, 3989, 3475, 1005, 1056, 2467, 2028, 2773, 2566, 19204, 1012]
Token strings: ['re', '-', 'token', '##ization', 'isn', "'", 't', 'always', 'one', 'word', 'per', 'token', '.']
[{'score': 0.6566779613494873, 'token': 6165, 'token_str': 'rates', 'sequence': 'the central bank decided to raise interest rates again.'}, {'score': 0.0663384422659874, 'token': 2320, 'token_str': 'once', 'sequence': 'the central bank decided to raise interest once again.'}, {'score': 0.05223919451236725, 'token': 3446, 'token_str': 'rate', 'sequence': 'the central bank decided to raise interest rate again.'}]


In [5]:
# only changed bank -> kitchen
changed_masked_text = f'The central kitchen decided to raise interest {unmasker.tokenizer.mask_token} again.'
changed_mask_results = unmasker(changed_masked_text, top_k=3)
if changed_mask_results is None:
    print('Complete the second input and compare its observed ranking.')
else:
    print(changed_masked_text)
    print('bank:   ', [(r['token_str'], round(r['score'], 3)) for r in mask_results])
    print('kitchen:', [(r['token_str'], round(r['score'], 3)) for r in changed_mask_results])

The central kitchen decided to raise interest [MASK] again.
bank:    [('rates', 0.657), ('once', 0.066), ('rate', 0.052)]
kitchen: [('once', 0.738), ('yet', 0.116), ('ever', 0.022)]


**Your interpretation:** The sentence has 7 words but 13 tokens. "Re-tokenization" became 4 tokens (re, -, token, ##ization), "isn't" became 3, and the full stop is a token too. Everything is lowercase because the model is uncased.

For "The central bank decided to raise interest [MASK] again." the top guess was rates (0.657). When I changed bank to kitchen, rates dropped out of the top 3 and once went from 0.066 to 0.738. So it was the word bank that made the model think about interest rates.

The top 3 scores don't sum to one because they are probabilities over the whole vocabulary (about 30 000 tokens) and we only look at the first 3.

## Task 3 · Stress-test a classifier (7 minutes)

Before running the model, label the four examples yourself as positive, negative, neutral or ambiguous. Then call the sentiment pipeline on the entire list, with input truncation enabled and a 128-token input cap. Compare your assessment with the model's available labels.

**Success:** four observed predictions with scores; one disagreement or ambiguous case discussed; one sentence explaining why a high score does not validate use on client tickets. If you agree with all four outputs, add an invented neutral or mixed-sentiment example and discuss the forced binary choice.

**Hint:** this checkpoint has no neutral output class. Inspect `sentiment.model.config.id2label`. A tiny hand-picked sample illustrates a failure mode; it is not a benchmark.

In [6]:
sentiment = load_pipeline('sentiment')
examples = [
    'The explanation was clear and useful.',
    'The application crashes every time I open it.',
    "The meeting starts at two o'clock.",
    'I loved the tutor, but the exercises were frustrating.',
]
my_labels = ['positive', 'negative', 'neutral', 'mixed']  # my guesses before running the model
student_predictions = sentiment(examples, truncation=True, max_length=128)
print('Available model labels:', sentiment.model.config.id2label)
if student_predictions is None or len(my_labels) != len(examples):
    print('Complete your judgments and the pipeline call.')
else:
    assert len(student_predictions) == len(examples)
    for text, human, model_result in zip(examples, my_labels, student_predictions):
        print({'text': text, 'my_judgment': human, 'model': model_result})

Device set to use cpu


sentiment: distilbert/distilbert-base-uncased-finetuned-sst-2-english @ 714eb0f
DistilBertForSequenceClassification / DistilBertTokenizerFast


Available model labels: {0: 'NEGATIVE', 1: 'POSITIVE'}
{'text': 'The explanation was clear and useful.', 'my_judgment': 'positive', 'model': {'label': 'POSITIVE', 'score': 0.9997910857200623}}
{'text': 'The application crashes every time I open it.', 'my_judgment': 'negative', 'model': {'label': 'NEGATIVE', 'score': 0.9997283816337585}}
{'text': "The meeting starts at two o'clock.", 'my_judgment': 'neutral', 'model': {'label': 'POSITIVE', 'score': 0.9768263697624207}}
{'text': 'I loved the tutor, but the exercises were frustrating.', 'my_judgment': 'mixed', 'model': {'label': 'NEGATIVE', 'score': 0.9912057518959045}}


**Your interpretation:** I labelled them positive, negative, neutral and mixed before running. The model agreed on the first two. The meeting sentence is just a fact, but the model said POSITIVE with 0.98, because it only has POSITIVE and NEGATIVE (see id2label), so it has to pick one. The tutor sentence is half good and half bad, and it said NEGATIVE 0.99.

So a high score doesn't mean the answer is right. The model was trained on movie reviews, not client tickets. Before using it on tickets I would label 100 or so real tickets by hand, including neutral ones, and check how often it is right.

## Task 4 · Bound generation and compare decoding (10 minutes)

Load the generation model and run the provided reporting helper. Write a call that generates **two sampled continuations with at most 15 new tokens each**. Use a seed of 42, one beam and deliberate sampling parameters. Request token IDs with `return_tensors=True`; pass the tokenizer's EOS ID as `pad_token_id` for this model.

Then generate **one greedy continuation** with the same prompt and token cap. Print all three continuations with both token counts and whitespace word counts.

**Success:** two sampled results, one greedy result, and each count at most 15. Your explanation says “maximum new tokens,” allows early stopping and does not require exactly 15 words, two complete sentences or different sampled texts.

**Hint:** inspect `max_new_tokens`, `num_return_sequences` and `do_sample`. Greedy decoding with one beam uses one return sequence. Tune `temperature` and `top_p` for sampling. For the greedy call, use neutral `temperature=1.0` to reset this v4 pipeline's inherited 0.7 default and avoid an ignored-parameter warning; it has no sampling effect when `do_sample=False`.

In [7]:
def describe_generations(results, prompt, tokenizer):
    """Count original generated IDs, rather than re-tokenizing decoded text."""
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    rows = []
    for result in results:
        all_ids = result['generated_token_ids']
        assert all_ids[:len(prompt_ids)] == prompt_ids
        added_ids = all_ids[len(prompt_ids):]
        # DistilGPT2 uses EOS as padding here; ignore padding after the first EOS.
        if tokenizer.eos_token_id in added_ids:
            added_ids = added_ids[:added_ids.index(tokenizer.eos_token_id) + 1]
        continuation = tokenizer.decode(added_ids, skip_special_tokens=True)
        rows.append({
            'continuation': continuation,
            'new_tokens_including_eos': len(added_ids),
            'whitespace_words': len(continuation.split()),
        })
    return rows

In [8]:
generator = load_pipeline('generation')
prompt = 'To prepare a useful training workshop, we should'
set_seed(SEED)
eos = generator.tokenizer.eos_token_id
# sampling, a bit less random than default
sampled_results = generator(prompt, max_new_tokens=15, do_sample=True, temperature=0.8, top_p=0.9,
                            num_beams=1, num_return_sequences=2, return_tensors=True, pad_token_id=eos)
# greedy, always takes the most likely token
greedy_results = generator(prompt, max_new_tokens=15, do_sample=False, temperature=1.0,
                           num_beams=1, num_return_sequences=1, return_tensors=True, pad_token_id=eos)

Device set to use cpu


generation: distilbert/distilgpt2 @ 2290a62
GPT2LMHeadModel / GPT2TokenizerFast


In [9]:
if sampled_results is None or greedy_results is None:
    print('Complete both generation calls, then rerun this check.')
else:
    sampled_rows = describe_generations(sampled_results, prompt, generator.tokenizer)
    greedy_rows = describe_generations(greedy_results, prompt, generator.tokenizer)
    assert len(sampled_rows) == 2
    assert len(greedy_rows) == 1
    assert all(1 <= row['new_tokens_including_eos'] <= 15 for row in sampled_rows + greedy_rows)
    print('Sampled:', sampled_rows)
    print('Greedy:', greedy_rows)

Sampled: [{'continuation': ' use the following three different tools to prepare for the new training.\n\n', 'new_tokens_including_eos': 15, 'whitespace_words': 12}, {'continuation': ' be using the tool set to prepare a new class. We can use the', 'new_tokens_including_eos': 15, 'whitespace_words': 14}]
Greedy: [{'continuation': ' be able to use the following techniques to prepare a useful training workshop.\n', 'new_tokens_including_eos': 15, 'whitespace_words': 13}]


**Your interpretation:** All three continuations used the full 15 tokens but only had 12 to 14 words, because newlines and punctuation are tokens too. The second sampled one even stops in the middle of a sentence. max_new_tokens is only an upper limit on tokens, it doesn't count words and generation can also stop earlier.

Greedy always takes the most likely next token, so it gives the same text every time, and here it basically repeated the prompt ("prepare a useful training workshop"). Sampling with temperature 0.8 and top_p 0.9 gave two different texts that depend on the seed.

Neither of them checks facts, they just write likely next words. For example the first sampled one says "the following three different tools" and no tools come after it.

If an app really needed exactly 15 words, I would generate more tokens than needed and then cut the text to 15 words in code, or generate again if it is too short.

## Task 5 · Make a model decision (5 minutes)

A team wants to use one of these checkpoints on a new task. Choose **one** scenario and write 4–6 sentences:

- Route short English customer-support messages to three departments.
- Suggest a continuation for public training copy.
- Extract named organizations from public event announcements.

Name a suitable task and model family, identify a mismatch with at least one of the core checkpoints, give one test case, and state one evaluation measure or review criterion. Also identify a case the system should hand to a human. You do not need to download another model.

**Success:** your recommendation follows the task and evidence; it does not use a sentiment score as routing accuracy or fluency as factual correctness.

**Your answer:** I picked routing support messages to three departments. It's a normal classification task, so I would fine-tune an encoder like DistilBERT with a 3 class head (for example billing, technical and account). The sentiment model doesn't fit, it only knows positive and negative, so an angry billing message and an angry crash report look the same to it. A good test case is "I was charged twice after the app crashed", because it belongs to two departments. I would check accuracy for each department and look at a confusion matrix on real labelled tickets. If the model is unsure, or the message is about fraud or legal stuff, a person should handle it.

## Optional investigation - choose one (10–15 minutes)

These preserve the original exploration topics. A model-card review is enough for the first pass; download only after checking size and language support. If you run a model, use an explicit model ID and record its commit revision, task, tokenizer, library version and actual output. If you do not run it, state **not executed** and explain the planned test.

| Option | Action | Evidence to submit |
|---|---|---|
| Another language | Find a text-generation model whose card supports a language you know. | Model URL, language, architecture, task, weight size, limitation and one bounded test prompt. |
| Cased BERT | Inspect [google-bert/bert-base-cased](https://huggingface.co/google-bert/bert-base-cased). Identify its mask marker and contrast cased/uncased tokenization. | The marker and source; if executed, the top three observed candidates for one input. Do not copy a prediction from a widget. |
| POS tagging | Inspect [vblagoje/bert-english-uncased-finetuned-pos](https://huggingface.co/vblagoje/bert-english-uncased-finetuned-pos) or another English POS checkpoint. | Task, at least three tag meanings, and why those labels differ from NER. Note any missing license/training information in the card. |
| Translation | Choose one directed language pair and inspect a suitable model card. | Source/target language, tokenizer requirements, model URL and a plan to check numbers, names and meaning. |
| Original advanced demos | Enable one optional section in the guided lab: zero-shot, NER, QA or summary. | Your observed result and one limitation demonstrated by changing the input. |

**Your investigation and execution status:** I picked Cased BERT, **executed** (cell below).

Model: [google-bert/bert-base-cased](https://huggingface.co/google-bert/bert-base-cased), revision cd5ef92, task fill-mask, tokenizer BertTokenizerFast, transformers 4.57.6. The mask marker is [MASK], I printed it from `cased.tokenizer.mask_token`.

I put the same sentence through both tokenizers. The uncased one lowercases everything, so Apple (the company) and New York look the same as apple and new york. The cased one keeps Apple and New York as whole tokens, but lowercase york gets split into yo and ##rk because it's rare written like that.

For "[MASK] shares fell after weak iPhone sales." the top 3 were The (0.168), Its (0.116) and Consumer (0.036). In lowercase, "shares of [MASK] fell after weak iphone sales." gave stock, shares and it. It never guessed Apple, so fill-mask just finds words that fit the sentence, it doesn't really know facts. For finance news where company names matter I would use a cased model, or better an NER model.

In [10]:
import transformers

cased = pipeline(task='fill-mask', model='google-bert/bert-base-cased', framework='pt', device=-1,
                 model_kwargs={'use_safetensors': True})
print('revision:', cased.model.config._commit_hash)
print('tokenizer:', type(cased.tokenizer).__name__, '| transformers', transformers.__version__)
print('mask marker:', cased.tokenizer.mask_token)

# same sentence in both tokenizers
sample = 'Apple sold apples in New York, not new york.'
print('uncased:', unmasker.tokenizer.tokenize(sample))
print('cased:  ', cased.tokenizer.tokenize(sample))

# does it guess the company?
for text in ['[MASK] shares fell after weak iPhone sales.',
             'shares of [MASK] fell after weak iphone sales.']:
    print(text, [(r['token_str'], round(r['score'], 3)) for r in cased(text, top_k=3)])

Some weights of the model checkpoint at google-bert/bert-base-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Device set to use cpu


revision: cd5ef92a9fb2f889e972770a36d4ed042daf221e
tokenizer: BertTokenizerFast | transformers 4.57.6
mask marker: [MASK]
uncased: ['apple', 'sold', 'apples', 'in', 'new', 'york', ',', 'not', 'new', 'york', '.']
cased:   ['Apple', 'sold', 'apples', 'in', 'New', 'York', ',', 'not', 'new', 'yo', '##rk', '.']


[MASK] shares fell after weak iPhone sales. [('The', 0.168), ('Its', 0.116), ('Consumer', 0.036)]
shares of [MASK] fell after weak iphone sales. [('stock', 0.114), ('shares', 0.049), ('it', 0.045)]


## Submission check

- Replace every required `TODO` with your own code or interpretation.
- Restart and run the notebook in order using the pinned kernel. Keep the printed versions.
- Keep your observed outputs; do not invent a plausible result for an unexecuted cell.
- Explain one failure or ambiguity even when the code runs successfully.
- Mark the optional investigation as executed or not executed.

**Assessment:** task/model mapping (2 points), token and mask evidence (2), sentiment interpretation (2), bounded generation and decoding comparison (3), model decision and limitation (1). Total: **10 points**. Optional work is for discussion, not required to achieve 10/10.

## Sources and attribution

The original notebook follows the Hugging Face course chapter [Transformers, what can they do?](https://huggingface.co/learn/llm-course/chapter1/3). This revision uses new classroom examples and adds setup, model selection, bounded generation, exercises and interpretation. This is **source attribution**, not a license grant; the course source and each model retain their own applicable licenses. Model weights are downloaded separately and are not distributed in this notebook.

Primary references checked for this revision:

- [Transformers 4.57.1 pipeline API](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines), including tasks, NER aggregation and output formats.
- [Transformers 4.57.1 generation parameters](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/text_generation), including sampling and token limits.
- [Installation, caching and offline mode](https://huggingface.co/docs/transformers/v4.57.1/en/installation).
- [Sentiment model card](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english) · [masked-language model card](https://huggingface.co/distilbert/distilbert-base-uncased) · [DistilGPT2 model card](https://huggingface.co/distilbert/distilgpt2).
- Weight-size evidence: [sentiment files](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/main), [mask-model files](https://huggingface.co/distilbert/distilbert-base-uncased/tree/main), [generation-model files](https://huggingface.co/distilbert/distilgpt2/tree/main). The figures are for one safetensors weight file, not the entire repository or runtime memory.

Model revision hashes in the code freeze the reviewed files. A fixed seed improves repeatability in a fixed environment; it does not guarantee bit-identical text across different hardware, library versions or decoding implementations. Inspect warnings; this notebook does not suppress them globally.